In [1]:
import pandas as pd
log = pd.read_csv("data/interim/filter_log.csv")
recent = log[log["filter_result"] == "released < 12 months ago"]
print(recent["release_date"].str[:4].value_counts().sort_index() if recent["release_date"].dtype == object else recent["release_date"].dt.year.value_counts().sort_index())
print(recent[["name", "release_date_raw", "age_days"]].sample(10, random_state=1))

release_date
2026    83
Name: count, dtype: int64
                                            name release_date_raw  age_days
27   Silk Collector: Tidy Up the Clothing Store!      Nov 6, 2026        12
115     Sidetrack — Tiny Trains for Your Desktop     Oct 22, 2026        27
76                                   Pomo Valley     Oct 30, 2026        19
92                                       Hearsay      Nov 3, 2026        15
66                          Teddy Store: Tidy Up     Nov 11, 2026         7
106                    DeskHugs: Desktop Friends      Nov 7, 2026        11
95                                          赛博拼豆     Oct 24, 2026        25
125                               Tiny Pet Salon      Nov 6, 2026        12
105                            Little Pagan Land      Nov 2, 2026        16
74                                 Only Axolotls      Nov 2, 2026        16


In [2]:
import pandas as pd
s = pd.read_csv("data/interim/sample.csv")
print(s["n_tags_matched"].value_counts().sort_index())

n_tags_matched
1    82
2    13
3     7
4     1
Name: count, dtype: int64


In [4]:
m = pd.read_csv("data/processed/game_metrics.csv")
print(m[(m.ach_median_pct == 0) | (m.ach_share_under_1pct > 0.5)][["name", "n_achievements_api", "ach_top_pct", "ach_median_pct", "age_days"]])

            name  n_achievements_api  ach_top_pct  ach_median_pct  age_days
81  Draw & Guess                  29          1.8             0.0      2077


In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect("data/steam.db")
print(pd.read_sql("SELECT name, n_achievements_api, ach_top_pct, ach_median_pct, age_days, owners FROM game_metrics WHERE low_signal_achievements = 1", con))

           name  n_achievements_api  ach_top_pct  ach_median_pct  age_days  \
0   Candy Blast                  18          1.7             1.0      3793   
1         Shift                  16          3.4             2.5      3746   
2  Mind Spheres                  11          4.7             3.2      3785   
3  Draw & Guess                  29          1.8             0.0      2077   
4       GoreBox                  31          4.8             1.1      1216   

                   owners  
0       50,000 .. 100,000  
1      100,000 .. 200,000  
2      200,000 .. 500,000  
3  2,000,000 .. 5,000,000  
4  1,000,000 .. 2,000,000  


In [3]:
import sqlite3, pandas as pd
from scipy import stats

con = sqlite3.connect("data/steam.db")
d = pd.read_sql("""
    SELECT b.activity_segment, b.ach_retention_ratio, b.ach_tail_ratio
    FROM analysis_base b JOIN game_metrics g ON g.appid = b.appid
    WHERE g.n_tags_matched >= 2
""", con)

print(d.groupby("activity_segment").agg(
    n=("ach_retention_ratio", "size"),
    median_retention=("ach_retention_ratio", "median"),
    median_tail=("ach_tail_ratio", "median")).round(3))

groups = [g["ach_retention_ratio"].dropna() for _, g in d.groupby("activity_segment")]
if all(len(g) > 1 for g in groups):
    print("Kruskal-Wallis p =", round(stats.kruskal(*groups).pvalue, 3))

                   n  median_retention  median_tail
activity_segment                                   
1_dormant          3             0.350        0.108
2_occasional      13             0.335        0.112
3_active           5             0.243        0.043
Kruskal-Wallis p = 0.379
